In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "5"

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from config import Config
import pandas as pd
import torch, torch.nn.functional as F

In [3]:
cfg = Config()

In [4]:
device = 'cuda'

In [5]:
forgt_set = pd.read_csv(cfg.forget_path)
retain_set = pd.read_csv(cfg.retain_path)

In [6]:
ben_var = forgt_set.loc[forgt_set['title'] == 'Benedetto Varchi'].reset_index(drop = True)
ret_var = retain_set.loc[retain_set['title'] == 'Benedetto Varchi'].reset_index(drop = True)

In [7]:
ben_var['set'] = 'forget'
ret_var['set'] = 'retain'
ben_var['type'] = 'forget'
ben_var = ben_var[['title', 'question', 'answer', 'set', 'type']]
ret_var['type'] = ret_var['type'].apply(lambda x: 'indirect' if x == 'domain' else 'direct')

In [8]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model_id, token = cfg.access_token)
tokenizer.pad_token = tokenizer.eos_token

### pre unlearning, forget and retain closeness for benedetto varchi

In [ ]:
model = AutoModelForCausalLM.from_pretrained(cfg.model_id, torch_dtype=torch.float16, token = cfg.access_token).to(device)

In [ ]:
def tokenize_text(q,a, tokenizer):
    q = tokenizer.apply_chat_template([{"role": "user", "content": q}]
                                     , tokenize = False, add_generation_prompt = True)
    full_text = q + a + tokenizer.eos_token
    return full_text

In [ ]:
def embed(texts, batch_size=16):
    """Return an (N, D) tensor of normalised embeddings."""
    outs = []
    for i in range(0, len(texts), batch_size):
        batch = tokenizer(texts[i:i+batch_size],
                    padding=True,
                    truncation=True,
                    return_tensors="pt").to(device)

        with torch.no_grad():
            hs = model(**batch, output_hidden_states=True).hidden_states[-1]  # (B,T,D)

        # mean-pool, masking out padding
        mask  = batch.attention_mask.unsqueeze(-1)                    # (B,T,1)
        pooled = (hs * mask).sum(1) / mask.sum(1)                     # (B,D)
        outs.append(F.normalize(pooled, p=2, dim=1).cpu())
    return torch.cat(outs)

In [79]:
def fuse(q, a):
    return f"{q.strip()} [SEP] {a.strip()}"

ben_var["text"] = ben_var.apply(lambda r: fuse(r.question, r.answer), axis=1)
ret_var["text"] = ret_var.apply(lambda r: fuse(r.question, r.answer), axis=1)

# ── 1-D.  Concatenate & tidy ──────────────────────────────────────────────
full = (pd.concat([ben_var, ret_var], ignore_index=True)
          .loc[:, ["title", "text", "set", "type"]])


In [80]:
full.head()

,title,text,set,type
0,Benedetto Varchi,What nationality was Benedetto Varchi? [SEP] I...,forget,forget
1,Benedetto Varchi,What professions did Benedetto Varchi have? [S...,forget,forget
2,Benedetto Varchi,Where was Benedetto Varchi born? [SEP] Florence,forget,forget
3,Benedetto Varchi,Who commissioned Benedetto Varchi to write a h...,forget,forget
4,Benedetto Varchi,When was Varchi's Storia fiorentina first publ...,forget,forget


In [82]:
full["embedding"] = list(embed(full["text"].tolist(), batch_size=8).numpy())

In [83]:
from sklearn.decomposition import PCA
import umap.umap_ as umap
import plotly.express as px
import numpy as np

In [84]:
emb   = np.vstack(full["embedding"].tolist()).astype("float32")
vec50 = PCA(n_components=50, random_state=0).fit_transform(emb)
proj  = umap.UMAP(n_components=2, metric="cosine", random_state=0, n_neighbors=50).fit_transform(vec50)
full[["x","y"]] = proj

/home/praveen/miniconda3/envs/emnlp/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning:

n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.



In [ ]:
import plotly.express as px

# ── choose whatever hues you like ───────────────────────────────
palette = {
    "direct":   "#1f77b4",   # blue
    "indirect": "#ff7f0e",   # orange
    "forget":   "#2ca02c"    # green (forget rows)
}

fig = px.scatter(
    full, x="x", y="y",
    color="type",                    # colours by subtype
    symbol="set",                    # shapes by retain/forget
    color_discrete_map=palette,
    category_orders={
        "type": ["direct", "indirect","forget"],
        "set":  ["retain", "forget"]
    },
    hover_data=["title", "text"],
    title="Benedetto varchi UMAP of "
)

fig.update_traces(marker=dict(size=7, opacity=0.85))
fig.write_html("embedding_map.html", include_plotlyjs="cdn")
print("Open embedding_map.html in any browser.")

Open embedding_map.html in any browser.


### pre unlearning, forget q & a closeness and retain q&a closeness

In [9]:
from peft import PeftModel
import uuid

In [10]:
model_pre = AutoModelForCausalLM.from_pretrained(cfg.model_id, torch_dtype=torch.bfloat16, token = cfg.access_token).to(device)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [11]:
def embed(texts, tok, mdl, batch=8):
    outs = []
    for i in range(0, len(texts), batch):
        batch_tok = tok(
            texts[i:i+batch], padding=True, truncation=True,
            return_tensors="pt"
        ).to(mdl.device)

        with torch.no_grad():
            hs = mdl(**batch_tok, output_hidden_states=True).hidden_states[-1].float()

        mask    = batch_tok.attention_mask.unsqueeze(-1)
        pooled  = (hs * mask).sum(1) / mask.sum(1)          # mean-pool
        outs.append(F.normalize(pooled, p=2, dim=1).cpu())
    return torch.cat(outs)

In [14]:
def explode_pairs(df):
    pair_ids = [uuid.uuid4().hex[:8] for _ in range(len(df))]
    return pd.DataFrame({
        "pair_id":  pair_ids + pair_ids,
        "role":     ["question"] * len(df) + ["answer"] * len(df),
        "text":     df["question"].tolist() + df["answer"].tolist(),
        "title":    df["title"].tolist()   * 2,
        "phase":    ["pre"] * len(df) * 2    
    })

forget2 = explode_pairs(ben_var)

In [15]:
forget2.head()

,pair_id,role,text,title,phase
0,b121f385,question,What nationality was Benedetto Varchi?,Benedetto Varchi,pre
1,fa9f0f82,question,What professions did Benedetto Varchi have?,Benedetto Varchi,pre
2,f93b9c14,question,Where was Benedetto Varchi born?,Benedetto Varchi,pre
3,7bfbb735,question,Who commissioned Benedetto Varchi to write a h...,Benedetto Varchi,pre
4,c5df3f9a,question,When was Varchi's Storia fiorentina first publ...,Benedetto Varchi,pre


In [16]:
forget2_pre  = forget2.copy()           
forget2_pre["embedding"] = list(
    embed(forget2_pre["text"].tolist(), tokenizer, model_pre).numpy()
)

In [18]:
forget2_pre.head(15)

,pair_id,role,text,title,phase,embedding
0,b121f385,question,What nationality was Benedetto Varchi?,Benedetto Varchi,pre,"[-0.023785222, -0.006875923, 0.012243439, 0.04..."
1,fa9f0f82,question,What professions did Benedetto Varchi have?,Benedetto Varchi,pre,"[-0.016553266, 0.0014623429, 0.008323668, 0.05..."
2,f93b9c14,question,Where was Benedetto Varchi born?,Benedetto Varchi,pre,"[-0.032573584, 0.01130415, 0.00027855925, 0.04..."
3,7bfbb735,question,Who commissioned Benedetto Varchi to write a h...,Benedetto Varchi,pre,"[-0.013000752, 0.0021260504, -0.015663465, 0.0..."
4,c5df3f9a,question,When was Varchi's Storia fiorentina first publ...,Benedetto Varchi,pre,"[-0.008197633, 0.011496132, -0.013290635, 0.03..."
5,6e28bf7b,question,Which work of Ezra Pound mentions Benedetto Va...,Benedetto Varchi,pre,"[-0.0019150771, -0.00042171386, 0.00611237, 0...."
6,13e08fc4,question,What was the main topic of Benedetto Varchi's ...,Benedetto Varchi,pre,"[-0.013245691, -0.0010684329, -0.012845744, 0...."
7,b121f385,answer,Italian,Benedetto Varchi,pre,"[0.0040764743, 0.012139279, -0.016025452, 0.03..."
8,fa9f0f82,answer,"Humanist, historian, poet",Benedetto Varchi,pre,"[-0.014087002, 0.0031588462, 0.01270481, 0.027..."
9,f93b9c14,answer,Florence,Benedetto Varchi,pre,"[-0.000134248, 0.014244418, -0.0034480537, 0.0..."


In [19]:
del model_pre
torch.cuda.empty_cache()

In [20]:
output_dir = '/home/praveen/theoden/emnlp25/outputs/cyclic_gd/checkpoint-900'

In [21]:
base_model = AutoModelForCausalLM.from_pretrained(cfg.model_id, torch_dtype=torch.bfloat16, token = cfg.access_token).to(device)
model_post = PeftModel.from_pretrained(base_model, output_dir, device_map="auto", torch_dtype=torch.bfloat16).to(device)
model_post = model_post.merge_and_unload()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [22]:
forget2_post = forget2_pre.copy()
forget2_post["phase"] = "post"
forget2_post["embedding"] = list(
    embed(forget2_post["text"].tolist(), tokenizer, model_post).numpy()
)

In [24]:
del model_post
torch.cuda.empty_cache()

In [23]:
both = pd.concat([forget2_pre, forget2_post], ignore_index=True)

In [29]:
both.head(15)

,pair_id,role,text,title,phase,embedding
0,b121f385,question,What nationality was Benedetto Varchi?,Benedetto Varchi,pre,"[-0.023785222, -0.006875923, 0.012243439, 0.04..."
1,fa9f0f82,question,What professions did Benedetto Varchi have?,Benedetto Varchi,pre,"[-0.016553266, 0.0014623429, 0.008323668, 0.05..."
2,f93b9c14,question,Where was Benedetto Varchi born?,Benedetto Varchi,pre,"[-0.032573584, 0.01130415, 0.00027855925, 0.04..."
3,7bfbb735,question,Who commissioned Benedetto Varchi to write a h...,Benedetto Varchi,pre,"[-0.013000752, 0.0021260504, -0.015663465, 0.0..."
4,c5df3f9a,question,When was Varchi's Storia fiorentina first publ...,Benedetto Varchi,pre,"[-0.008197633, 0.011496132, -0.013290635, 0.03..."
5,6e28bf7b,question,Which work of Ezra Pound mentions Benedetto Va...,Benedetto Varchi,pre,"[-0.0019150771, -0.00042171386, 0.00611237, 0...."
6,13e08fc4,question,What was the main topic of Benedetto Varchi's ...,Benedetto Varchi,pre,"[-0.013245691, -0.0010684329, -0.012845744, 0...."
7,b121f385,answer,Italian,Benedetto Varchi,pre,"[0.0040764743, 0.012139279, -0.016025452, 0.03..."
8,fa9f0f82,answer,"Humanist, historian, poet",Benedetto Varchi,pre,"[-0.014087002, 0.0031588462, 0.01270481, 0.027..."
9,f93b9c14,answer,Florence,Benedetto Varchi,pre,"[-0.000134248, 0.014244418, -0.0034480537, 0.0..."


In [25]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def qa_dist(grp):
    q = np.stack(grp.loc[grp.role=="question", "embedding"])
    a = np.stack(grp.loc[grp.role=="answer",   "embedding"])
    return 1 - cosine_similarity(q, a)[0,0]

dist = (both.groupby(["pair_id", "phase"])
             .apply(qa_dist)
             .rename("cos_dist")
             .reset_index())

# pivot wide: pre | post | delta
dist_wide = (dist.pivot(index="pair_id", columns="phase", values="cos_dist")
                 .assign(delta=lambda d: d["post"] - d["pre"]))

/tmp/ipykernel_3041083/2743693440.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(qa_dist)


In [26]:
import numpy as np
mean_delta = (dist_wide["delta"]).mean()
print(f"mean Δdistance = {mean_delta:+.4f}")      # expect a small negative number

improvement_ratio = (dist_wide["delta"] < 0).mean()
print(f"{improvement_ratio:.0%} of pairs got closer")


mean Δdistance = +0.0365
14% of pairs got closer


In [27]:
import plotly.express as px

# histogram of change
px.histogram(dist_wide, x="delta", nbins=30,
             title="Δ Cosine distance (post – pre)").add_vline(x=0)

# scatter pre vs post
fig = px.scatter(dist_wide, x="pre", y="post",
                 title="Per-pair Q–A distance shift for pre unlearning and cyclic gd")
fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1,
              line=dict(dash="dash"))
fig.update_layout(xaxis_title="pre", yaxis_title="post")
fig.show()

In [36]:
import plotly.express as px

# histogram of change
px.histogram(dist_wide, x="delta", nbins=30,
             title="Δ Cosine distance (post – pre)").add_vline(x=0)

# scatter pre vs post
fig = px.scatter(dist_wide, x="pre", y="post",
                 title="Per-pair Q–A distance shift for pre unlearning and gradient asent")
fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1,
              line=dict(dash="dash"))
fig.update_layout(xaxis_title="pre", yaxis_title="post")
fig.show()

In [21]:
import plotly.express as px

# histogram of change
px.histogram(dist_wide, x="delta", nbins=30,
             title="Δ Cosine distance (post – pre)").add_vline(x=0)

# scatter pre vs post
fig = px.scatter(dist_wide, x="pre", y="post",
                 title="Per-pair Q–A distance shift for pre unlearning and melu unlearning gradient")
fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1,
              line=dict(dash="dash"))
fig.update_layout(xaxis_title="pre", yaxis_title="post")
fig.show()

In [20]:
import plotly.express as px

# histogram of change
px.histogram(dist_wide, x="delta", nbins=30,
             title="Δ Cosine distance (post – pre)").add_vline(x=0)

# scatter pre vs post
fig = px.scatter(dist_wide, x="pre", y="post",
                 title="Per-pair Q–A distance shift for pre unlearning and 1:1 seq unlearning gradient")
fig.add_shape(type="line", x0=0, y0=0, x1=1, y1=1,
              line=dict(dash="dash"))
fig.update_layout(xaxis_title="pre", yaxis_title="post")
fig.show()